# Tapis Postgres migration helper

This notebook handles the Postgres dump and restore workflow separately from the pod and volume migration.


## 1. Install and Import Libraries

Load Tapis helpers, Postgres tooling helpers, and environment variables from `.env.example`.


In [1]:
import os
import json
import shutil
import subprocess
import getpass
from pathlib import Path
from dotenv import load_dotenv
from tapipy.tapis import Tapis

load_dotenv(Path('.env.example'))

def create_tapis_client(base_url, tenant_id, username, password):
    client = Tapis(base_url=base_url, tenant_id=tenant_id, username=username, password=password)
    client.get_tokens()
    return client

def tapis_to_jsonable(value):
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, dict):
        return {k: tapis_to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [tapis_to_jsonable(v) for v in value]
    if hasattr(value, '__dict__'):
        return {
            k: tapis_to_jsonable(v)
            for k, v in vars(value).items()
            if not k.startswith('_') and not callable(v)
        }
    return str(value)

def resource_items(response):
    if isinstance(response, list):
        return response
    if isinstance(response, dict):
        return response.get('result', response)
    result = getattr(response, 'result', None)
    return result if result is not None else response

def save_json(path, data):
    Path(path).write_text(json.dumps(tapis_to_jsonable(data), indent=2))

def require_command(name, env_var=None):
    override = os.getenv(env_var, '') if env_var else ''
    if override:
        override_path = Path(override).expanduser()
        if override_path.exists():
            return str(override_path)
        raise RuntimeError(f'{env_var} is set but the file does not exist: {override_path}')
    resolved = shutil.which(name)
    if resolved:
        return resolved
    extra = f' Set {env_var} to the full path of the binary.' if env_var else ''
    raise RuntimeError(f'{name} is not installed or not on PATH.{extra}')

def env_override(name, placeholder_values=()):
    value = os.getenv(name, '')
    if not value:
        return None
    if value in placeholder_values:
        return None
    return value

def external_pods_port(host, fallback_port):
    if host and '.pods.' in str(host):
        return 443
    return fallback_port

print('Libraries imported successfully.')


Libraries imported successfully.


## 2. Authenticate to Both Tenants

Create source and destination Tapis clients so the notebook can inspect the live Postgres pod definitions.


In [2]:
source_config = {
    'base_url': 'https://tacc.tapis.io',
    'tenant_id': 'tacc',
    'username': os.getenv('SOURCE_TAPIS_USERNAME', ''),
    'password': os.getenv('SOURCE_TAPIS_PASSWORD', ''),
}
dest_config = {
    'base_url': 'https://portals.tapis.io',
    'tenant_id': 'portals',
    'username': os.getenv('DEST_TAPIS_USERNAME', '') or os.getenv('SOURCE_TAPIS_USERNAME', ''),
    'password': os.getenv('DEST_TAPIS_PASSWORD', '') or os.getenv('SOURCE_TAPIS_PASSWORD', ''),
}

if not source_config['username']:
    source_config['username'] = input('Source Tapis username: ')
if not source_config['password']:
    source_config['password'] = getpass.getpass('Source Tapis password: ')
if not dest_config['username']:
    dest_config['username'] = input('Destination Tapis username: ')
if not dest_config['password']:
    dest_config['password'] = getpass.getpass('Destination Tapis password: ')

source_client = create_tapis_client(**source_config)
dest_client = create_tapis_client(**dest_config)
print('Authenticated to source tenant:', source_config['base_url'])
print('Authenticated to destination tenant:', dest_config['base_url'])


Authenticated to source tenant: https://tacc.tapis.io
Authenticated to destination tenant: https://portals.tapis.io


## 3. Fetch Pods and Identify the Database Pods

Read pod definitions from both tenants and select the exact database pods to migrate.


In [3]:
TARGET_POSTGRES_POD_IDS = ['fluxpostgres', 'vitalpostgres', 'upstreampostgres']

def pods_by_id(pods):
    indexed = {}
    for pod in resource_items(pods):
        pod_data = tapis_to_jsonable(pod)
        pod_id = pod_data.get('pod_id')
        if pod_id:
            indexed[pod_id] = pod_data
    return indexed

source_pods = source_client.pods.list_pods()
dest_pods = dest_client.pods.list_pods()
save_json('source_pods.json', source_pods)
save_json('destination_pods.json', dest_pods)

source_pods_index = pods_by_id(source_pods)
dest_pods_index = pods_by_id(dest_pods)

missing_source = [pod_id for pod_id in TARGET_POSTGRES_POD_IDS if pod_id not in source_pods_index]
missing_dest = [pod_id for pod_id in TARGET_POSTGRES_POD_IDS if pod_id not in dest_pods_index]

if missing_source:
    raise RuntimeError(f'Missing source database pods: {missing_source}')
if missing_dest:
    raise RuntimeError(f'Missing destination database pods: {missing_dest}')

source_postgres_pods = {pod_id: source_pods_index[pod_id] for pod_id in TARGET_POSTGRES_POD_IDS}
dest_postgres_pods = {pod_id: dest_pods_index[pod_id] for pod_id in TARGET_POSTGRES_POD_IDS}
print('Source database pods:', list(source_postgres_pods))
print('Destination database pods:', list(dest_postgres_pods))


Source database pods: ['fluxpostgres', 'vitalpostgres', 'upstreampostgres']
Destination database pods: ['fluxpostgres', 'vitalpostgres', 'upstreampostgres']


## 4. Build Database Connection Settings

Derive source and destination connection settings for each database pod from the live pod definitions, with env-var overrides allowed.


In [4]:
def build_db_config(source_pod, dest_pod):
    source_env = source_pod.get('environment_variables', {}) or {}
    source_network = source_pod.get('networking', {}).get('default', {}) or {}
    dest_env = dest_pod.get('environment_variables', {}) or {}
    dest_network = dest_pod.get('networking', {}).get('default', {}) or {}
    source_host = env_override('PG_HOST', ('source-db-host.example.com',)) or source_network.get('url', 'source-db-host.example.com')
    source_default_port = external_pods_port(source_host, source_network.get('port', 5432))
    source_db = {
        'host': source_host,
        'port': int(env_override('PG_PORT') or source_default_port),
        'dbname': env_override('PG_DB', ('mydb',)) or source_env.get('POSTGRES_DB', 'postgres'),
        'user': env_override('PG_USER', ('postgres',)) or source_env.get('POSTGRES_USER', 'postgres'),
        'password': env_override('PG_PASSWORD') or source_env.get('POSTGRES_PASSWORD', ''),
    }
    dest_host = env_override('DEST_PG_HOST') or dest_network.get('url', source_db['host'])
    dest_default_port = external_pods_port(dest_host, dest_network.get('port', source_db['port']))
    dest_db = {
        'host': dest_host,
        'port': int(env_override('DEST_PG_PORT') or dest_default_port),
        'dbname': env_override('DEST_PG_DB') or dest_env.get('POSTGRES_DB', source_db['dbname']),
        'user': env_override('DEST_PG_USER') or dest_env.get('POSTGRES_USER', source_db['user']),
        'password': env_override('DEST_PG_PASSWORD') or dest_env.get('POSTGRES_PASSWORD', source_db['password']),
    }
    return source_db, dest_db

db_configs = {}
for pod_id in TARGET_POSTGRES_POD_IDS:
    source_db, dest_db = build_db_config(source_postgres_pods[pod_id], dest_postgres_pods[pod_id])
    if not source_db['password']:
        source_db['password'] = getpass.getpass(f'Source Postgres password for {pod_id}: ')
    if not dest_db['password']:
        dest_db['password'] = getpass.getpass(f'Destination Postgres password for {pod_id}: ')
    db_configs[pod_id] = {'source': source_db, 'dest': dest_db}
    print(f"{pod_id}: {source_db['host']} -> {dest_db['host']}")


fluxpostgres: fluxpostgres.pods.portals.tapis.io -> fluxpostgres.pods.portals.tapis.io
vitalpostgres: vitalpostgres.pods.portals.tapis.io -> vitalpostgres.pods.portals.tapis.io
upstreampostgres: upstreampostgres.pods.portals.tapis.io -> upstreampostgres.pods.portals.tapis.io


## 5. Dump the Source Database

Use `pg_dump` to export each source database to a local dump file.


In [5]:
pg_dump_bin = require_command('pg_dump', 'PG_DUMP_BIN')
dump_paths = {}
for pod_id, config in db_configs.items():
    source_db = config['source']
    dump_path = Path(f'{pod_id}_source_pg_dump.sql')
    dump_cmd = [
        pg_dump_bin,
        '-h', source_db['host'],
        '-p', '443',
        '-U', source_db['user'],
        '-Fc',
        '-f', str(dump_path),
        source_db['dbname'],
    ]
    print('Running pg_dump:', ' '.join(dump_cmd))
    try:
        subprocess.run(
            dump_cmd,
            check=True,
            text=True,
            capture_output=True,
            env={**os.environ, 'PGPASSWORD': source_db['password']},
        )
    except subprocess.CalledProcessError as exc:
        print(exc.stdout)
        print(exc.stderr)
        raise
    dump_paths[pod_id] = dump_path
    print('Wrote source Postgres dump to', dump_path)


Running pg_dump: /usr/local/bin/pg_dump -h fluxpostgres.pods.portals.tapis.io -p 443 -U fastapi_traefik -Fc -f fluxpostgres_source_pg_dump.sql fastapi_traefik
Wrote source Postgres dump to fluxpostgres_source_pg_dump.sql
Running pg_dump: /usr/local/bin/pg_dump -h vitalpostgres.pods.portals.tapis.io -p 443 -U fastapi_traefik -Fc -f vitalpostgres_source_pg_dump.sql fastapi_traefik
Wrote source Postgres dump to vitalpostgres_source_pg_dump.sql
Running pg_dump: /usr/local/bin/pg_dump -h upstreampostgres.pods.portals.tapis.io -p 443 -U fastapi_traefik -Fc -f upstreampostgres_source_pg_dump.sql fastapi_traefik
Wrote source Postgres dump to upstreampostgres_source_pg_dump.sql


## 6. Restore the Dump to the Destination Database

Use `pg_restore` to import each dump into the matching destination database.


In [6]:
pg_restore_bin = require_command('pg_restore', 'PG_RESTORE_BIN')
for pod_id, config in db_configs.items():
    dest_db = config['dest']
    dump_path = dump_paths[pod_id]
    restore_cmd = [
        pg_restore_bin,
        '-h', dest_db['host'],
        '-p', '443',
        '-U', dest_db['user'],
        '-d', dest_db['dbname'],
        '-c',
        '--if-exists',
        str(dump_path),
    ]
    print('Running pg_restore:', ' '.join(restore_cmd))
    try:
        subprocess.run(
            restore_cmd,
            check=True,
            text=True,
            capture_output=True,
            env={**os.environ, 'PGPASSWORD': dest_db['password']},
        )
    except subprocess.CalledProcessError as exc:
        print(exc.stdout)
        print(exc.stderr)
        raise
    print(f'Restored destination Postgres for {pod_id} from', dump_path)


Running pg_restore: /usr/local/bin/pg_restore -h fluxpostgres.pods.portals.tapis.io -p 443 -U fastapi_traefik -d fastapi_traefik -c --if-exists fluxpostgres_source_pg_dump.sql
Restored destination Postgres for fluxpostgres from fluxpostgres_source_pg_dump.sql
Running pg_restore: /usr/local/bin/pg_restore -h vitalpostgres.pods.portals.tapis.io -p 443 -U fastapi_traefik -d fastapi_traefik -c --if-exists vitalpostgres_source_pg_dump.sql
Restored destination Postgres for vitalpostgres from vitalpostgres_source_pg_dump.sql
Running pg_restore: /usr/local/bin/pg_restore -h upstreampostgres.pods.portals.tapis.io -p 443 -U fastapi_traefik -d fastapi_traefik -c --if-exists upstreampostgres_source_pg_dump.sql
Restored destination Postgres for upstreampostgres from upstreampostgres_source_pg_dump.sql


## 7. Verify the Restored Database

Connect to each destination database and inspect the schema after restore.


In [7]:
import psycopg2

for pod_id, config in db_configs.items():
    dest_db = config['dest']
    with psycopg2.connect(
        host=dest_db['host'],
        port='443',
        dbname=dest_db['dbname'],
        user=dest_db['user'],
        password=dest_db['password'],
    ) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT count(*) FROM information_schema.tables WHERE table_schema = 'public'")
            table_count = cur.fetchone()[0]
            cur.execute("SELECT tablename FROM pg_catalog.pg_tables WHERE schemaname = 'public' ORDER BY tablename LIMIT 10")
            sample_tables = [row[0] for row in cur.fetchall()]
    print(f'{pod_id} destination public table count:', table_count)
    print(f'{pod_id} sample destination tables:', sample_tables)


fluxpostgres destination public table count: 13
fluxpostgres sample destination tables: ['alembic_version', 'campaigns', 'measurements', 'metadata_schema', 'sensor_statistics', 'sensorobjects', 'sensors', 'spatial_ref_sys', 'stations', 'upload_file_events']
vitalpostgres destination public table count: 13
vitalpostgres sample destination tables: ['alembic_version', 'campaigns', 'measurements', 'metadata_schema', 'sensor_statistics', 'sensorobjects', 'sensors', 'spatial_ref_sys', 'stations', 'upload_file_events']
upstreampostgres destination public table count: 13
upstreampostgres sample destination tables: ['alembic_version', 'campaigns', 'measurements', 'metadata_schema', 'sensor_statistics', 'sensorobjects', 'sensors', 'spatial_ref_sys', 'stations', 'upload_file_events']
